In [34]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from langchain.agents.middleware import wrap_model_call


load_dotenv()

True

In [2]:
embeddings= HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1646.62it/s]


In [3]:
django_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="django_docs",
    embedding_function=embeddings
)

python_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="python_scripts",
    embedding_function=embeddings
)

In [4]:
django_db= django_vectorstore.get()
python_db=python_vectorstore.get()

django_splits=[
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(django_db["documents"], django_db["metadatas"])
]

python_splits = [
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(python_db["documents"], python_db["metadatas"])
]

django_retriever= django_vectorstore.as_retriever(search_kwargs={"k":4})
python_retriever= python_vectorstore.as_retriever(search_kwargs={"k":4})


all_splits= django_splits + python_splits
print(f"Loaded {len(all_splits)} total splits ({len(django_splits)} Django docs + {len(python_splits)} Python codebase).")
bm25_retriever= BM25Retriever.from_documents(all_splits)


Loaded 6361 total splits (5110 Django docs + 1251 Python codebase).


In [5]:
bm25_retriever.k=8

In [6]:
#Creating hybrid retriever
hybrid_retriever= EnsembleRetriever(
    retrievers=[django_retriever, python_retriever, bm25_retriever],
    weights=[0.4,0.3,0.3]#40% django sementic search, 30% python sementic and keyword
    
)

In [25]:
#Reranker block of code
reranker= CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2819.35it/s]


In [26]:
@tool
def retrieve_django_content(query: str) -> str:
    """
    Search the Django knowledge base for documentation, code examples,
    debugging information, and practical examples.
    Use this tool to gain more context and information needed to answer the users question
    """
    print("Using retriever......")
    docs = hybrid_retriever.invoke(query)

    print("Reranking......")
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )
    top_docs = scored_docs[:3]

    return "\n\n".join(doc.page_content for doc, score in top_docs)

In [27]:
#Constructing Agent and memory
llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
memory=MemorySaver()



In [40]:
#Trimming users messages to preserve context window
@wrap_model_call
def limit_history(request, handler):
    messages = request.state["messages"]

    recent_messages = messages[-2:]

    request.state["messages"] = recent_messages

    return handler(request)

In [45]:
agent= create_agent(
    model=llm,
    tools=[retrieve_django_content],
    system_prompt="""
    You are a Django codebase reviewer with 10+ years of experience reviewing code.
    Your job:
    1. Answer questions using ONLY the retrieved context
    2. Keep responses SHORT and DIRECT
    3. Never add information outside the retrieved documents
    4. Format answers clearly with bullet points when listing multiple items

    Response Rules (CRITICAL):
    - Maximum 2-3 sentences for simple questions
    - Maximum 1 paragraph (3-4 sentences) for complex questions
    - Use bullet points ONLY when the user asks for a list
    - Never include meta-commentary like "Based on the retrieved documents..."
    - If context doesn't answer the question, say: "I don't have this information in my knowledge base"

    When answering:
    - Lead with the direct answer first
    - Add only essential context if needed
    - Stop writing as soon as you've answered the question

    Format example:
    Q: What is X?
    A: X is [direct definition]. [One additional detail if relevant].

    NEVER:
    - Explain your retrieval process
    - Give multiple interpretations
    - Add "In conclusion..." or similar filler
    - Ramble or over-explain
    """,
    middleware=[limit_history],
    checkpointer=memory
)

#Memory
config = {"configurable": {"thread_id": "test-4"}}

In [46]:
#Creatin User Interface
while True:
    question= input("Any Question about django: ").strip()
    if question.lower() in ["exit", "quit", "q"]:
        print("See you soon...")
        break
    if not question:
        continue

    result= agent.invoke({
        "messages": [HumanMessage(content=question)]
    }, config=config)

    print("\n Final Result")
    print(result["messages"][-1].content)


 Final Result
- **Authentication & role check** – Using `@login_required` and `@user_passes_test` delegates security to Django’s built‑in decorators, guaranteeing the user is authenticated before any view logic runs and preventing accidental bypasses that manual `if not request.user.is_authenticated` checks can cause.  
- **Query optimisation** – `select_related('company')` and `.only(...)` fetch the agent and its company in a single query and limit columns, while `count()` replaces the Python `set(...).len()` pattern, eliminating unnecessary data transfer and N+1 queries.  
- **Date filtering** – Replacing `date_created=timezone.now().date()` with `date_created__date=timezone.localdate()` (and similarly for appointments) ensures correct timezone‑aware filtering on `DateTimeField`s and avoids missed rows.  
- **Error handling** – Catching only the expected exceptions (`AgentInformation.DoesNotExist`, `CompanyInformation.DoesNotExist`) and importing `traceback` prevents the broad `exce